In [ ]:
import numpy as np
from gfsupg.solver import CartesianGeometry, FiniteElement1D, Scipy2DFEM
from gfsupg.solver import DeC, DeCSpaceTimeSUPGSolver
from gfsupg.problem import LinearAcoustic2D
from gfsupg.plotting import *

import matplotlib.pyplot as plt

In [ ]:

#problem.T_fin = 1.
order=3

FEM1Dx = FiniteElement1D(order-1,"gaussLobatto","gaussLobatto")
FEM1Dy = FiniteElement1D(order-1,"gaussLobatto","gaussLobatto")
dec = DeC((order+1)//2,order,"gaussLobatto")


In [ ]:
problem = LinearAcoustic2D("SG")#("source_vortex")#("SG")#("coriolis_vortex")

In [ ]:

Ns = np.array([10,10], dtype=np.int32)

geom = CartesianGeometry(problem.xL,problem.xR, Ns, problem.geometry_folder, BC=problem.BC)

In [ ]:
FEM2D = Scipy2DFEM(geom,FEM1Dx, FEM1Dy, folder=problem.folderName)

In [ ]:
solver = DeCSpaceTimeSUPGSolver(problem, FEM2D, dec)

In [ ]:
q_save, tt_save, comp_time, err, err_vertex  = solver.solve(with_error = True, with_error_vertex = True, GF=False)

In [ ]:
solver.ic_vect

plot_one_sol(problem, FEM2D, solver.ic_vect)

In [ ]:
q_new = FEM2D.divfree_projection_optimization(solver.ic_vect, problem)


In [ ]:
error = dict()
for var in problem.vars:
    error[var] = np.abs(solver.ic_vect[var]-q_new[var])
fig = plot_one_sol(problem, FEM2D, solver.ic_vect, levels=100)
fig.suptitle("IC")
fig = plot_one_sol(problem, FEM2D, q_new, levels=100)
fig.suptitle("optimized IC")
fig = plot_one_sol(problem, FEM2D, error, levels=100)
fig.suptitle("Error with respect exact IC")

if problem.source is not None: 
    DIV = FEM2D.compute_discrete_divergence_residual(q_new, FEM2D.evaluate_function(problem.source["p"]))
else:
    DIV = FEM2D.compute_discrete_divergence(q_new)

DIV = FEM2D.vect_to_mat(DIV)

X = FEM2D.xx_mat[0]
Y = FEM2D.xx_mat[1]


sources = FEM2D.compute_sources(q_new,problem)
residual = FEM2D.compute_GF_residual(q_new,sources)

plt.figure(figsize=(6,5))
plt.contourf(X[1:-1,1:-1],Y[1:-1,1:-1],DIV[1:-1,1:-1],levels=100)
plt.axis("equal")
plt.title("Divergence of optimized IC ord %d N %04d no boundary"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0]))
plt.colorbar()
plt.tight_layout()

plt.figure(figsize=(6,5))
plt.contourf(X,Y,DIV,levels=100)
plt.axis("equal")
plt.title("Divergence of optimized IC ord %d N %04d"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0]))
plt.colorbar()
plt.tight_layout()


fig = plot_one_sol(problem, FEM2D, residual, levels=100)
fig.suptitle("Residuals")


fig = plot_one_sol(problem, FEM2D, residual, levels=100, no_boundaries =True)
fig.suptitle("Residuals no boundaries")





In [ ]:
error = dict()
for var in problem.vars:
    error[var] = np.abs(solver.ic_vect[var]-q_new[var])
fig = plot_one_sol(problem, FEM2D, solver.ic_vect, levels=100)
fig.suptitle("IC")
fig = plot_one_sol(problem, FEM2D, q_new, levels=100)
fig.suptitle("optimized IC")
fig = plot_one_sol(problem, FEM2D, error, levels=100)
fig.suptitle("Error with respect exact IC")

if problem.source is not None: 
    DIV = FEM2D.compute_discrete_divergence_residual(q_new, FEM2D.evaluate_function(problem.source["p"]))
else:
    DIV = FEM2D.compute_discrete_divergence(q_new)

DIV = FEM2D.vect_to_mat(DIV)

X = FEM2D.xx_mat[0]
Y = FEM2D.xx_mat[1]


sources = FEM2D.compute_sources(q_new,problem)
residual = FEM2D.compute_GF_residual(q_new,sources)

plt.figure(figsize=(6,5))
plt.contourf(X[1:-1,1:-1],Y[1:-1,1:-1],DIV[1:-1,1:-1],levels=100)
plt.axis("equal")
plt.title("Divergence of optimized IC ord %d N %04d no boundary"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0]))
plt.colorbar()
plt.tight_layout()

plt.figure(figsize=(6,5))
plt.contourf(X,Y,DIV,levels=100)
plt.axis("equal")
plt.title("Divergence of optimized IC ord %d N %04d"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0]))
plt.colorbar()
plt.tight_layout()


fig = plot_one_sol(problem, FEM2D, residual, levels=100)
fig.suptitle("Residuals")


fig = plot_one_sol(problem, FEM2D, residual, levels=100, no_boundaries =True)
fig.suptitle("Residuals no boundaries")





In [ ]:
stop()

# Mario

In [ ]:
q_new = FEM2D.divfree_projection_integration(solver.ic_vect, problem.ics, problem)


In [ ]:
error = dict()
for var in problem.vars:
    error[var] = solver.ic_vect[var]-q_new[var]
fig = plot_one_sol(problem, FEM2D, solver.ic_vect, levels=100)
fig.suptitle("IC")
fig = plot_one_sol(problem, FEM2D, q_new, levels=100)
fig.suptitle("projected IC")
fig = plot_one_sol(problem, FEM2D, error, levels=100)
fig.suptitle("Error with respect exact IC")

DIV = FEM2D.compute_discrete_divergence(q_new)

DIV = FEM2D.vect_to_mat(DIV)

X = FEM2D.xx_mat[0]
Y = FEM2D.xx_mat[1]


fig,axs=plt.subplots(1,2,figsize=(10,4))
cnt=axs[0].contourf(X[1:-1,1:-1],Y[1:-1,1:-1],DIV[1:-1,1:-1],levels=100)
fig.colorbar(cnt, ax=axs[0])

cnt=axs[1].contourf(X,Y,DIV,levels=100)
fig.colorbar(cnt, ax=axs[1])
axs[0].axis("equal")
axs[1].axis("equal")
fig.suptitle("Divergence of integrated IC ord %d N %04d"%(FEM2D.FEM1Dx.degree+1,FEM2D.geom.N_elem_dir[0]))
axs[0].set_title("without boundary")
axs[1].set_title("with boundary")
plt.tight_layout()

In [ ]:
stop()

# Davide approach

In [ ]:

from scipy.optimize import LinearConstraint, minimize
from scipy.sparse import hstack, identity, vstack

def divfree_projection_optimization(FEM2D, IC_vect):
    div_oper = hstack([FEM2D.operator["IDx_tilde"],FEM2D.operator["IDy_tilde"]] )

    linear_constraint = LinearConstraint(div_oper , np.zeros(div_oper.shape[0]), np.zeros(div_oper.shape[0]))


    u = IC_vect["u"]
    v = IC_vect["v"]

    u_ic = np.concatenate([u,v])

    def target_function(u):
        return np.sum((u-u_ic)**2)
    def target_function_dir(u):
        return 2.*(u-u_ic)
    def target_function_hess(u):
        return 2.*identity(len(u))

    res = minimize(target_function, u_ic, method='trust-constr', jac=target_function_dir, hess=target_function_hess,
                constraints=[linear_constraint],
                options={'verbose': 1})
    
    q_vec = dict()
    q_vec["u"]       = res.x[:len(u)]
    q_vec["v"]       = res.x[len(u):]
    q_vec["p"]       = IC_vect["p"]
    return q_vec


q_vec = divfree_projection_optimization(FEM2D, solver.ic_vect)

In [ ]:
plot_one_sol(problem, FEM2D, solver.ic_vect)
plot_one_sol(problem, FEM2D, q_vec)

In [ ]:
p_ic = np.copy(solver.ic_vect["p"])
def target_function(p):
    return np.sum((p-p_ic)**2)
def target_function_dir(p):
    return 2.*(p-p_ic)
def target_function_hess(p):
    return 2.*identity(len(p))
cor = problem.coriolis/problem.c
op = FEM2D.operator
dxdy = geom.dx[0]*geom.dx[1]
linear_constraint_v =\
      LinearConstraint(op["IDx"]/dxdy,\
                    (cor)/dxdy*op["mass_tilde_x"]@q_vec["v"],\
                    (cor)/dxdy*op["mass_tilde_x"]@q_vec["v"])
linear_constraint_u =\
      LinearConstraint(op["IDy"]/dxdy,\
                    -(cor)/dxdy*op["mass_tilde_y"]@ q_vec["u"],\
                    -(cor)/dxdy*op["mass_tilde_y"]@ q_vec["u"])

average_constraint = \
    LinearConstraint(op["IDx"]/dxdy+op["IDy"]/dxdy,\
                     (cor)/dxdy*op["mass_tilde_x"]@q_vec["v"]-(cor)/dxdy*op["mass_tilde_y"]@ q_vec["u"],\
                     (cor)/dxdy*op["mass_tilde_x"]@q_vec["v"]-(cor)/dxdy*op["mass_tilde_y"]@ q_vec["u"]\
                     )

rhs = np.concatenate([ (cor)/dxdy*op["mass_tilde_x"]@q_vec["v"],\
                      -(cor)/dxdy*op["mass_tilde_y"]@q_vec["u"]])

all_constraints = LinearConstraint(\
                vstack([op["IDx"]/dxdy,op["IDy"]/dxdy]),\
                rhs, rhs)



In [ ]:
a,b  = linear_constraint_u.residual(p_ic)
print(np.max(np.abs(a)),np.max(np.abs(b)))
a,b  = linear_constraint_v.residual(p_ic)
print(np.max(np.abs(a)),np.max(np.abs(b)))
a,b  = all_constraints.residual(p_ic)
print(np.max(np.abs(a)),np.max(np.abs(b)))

In [ ]:
fig,ax = plt.subplots(1,1)

zz0 = op["IDx_tilde"]@q_vec["p"]-(cor)*op["mass_tilde_x"]@ q_vec["v"]
zz0=zz0/dxdy
plot_sol(FEM2D, zz0, ax, fig)

fig,ax = plt.subplots(1,1)
zz = op["IDy_tilde"]@q_vec["p"]+(cor)*op["mass_tilde_y"]@ q_vec["u"]
zz=zz/dxdy
plot_sol(FEM2D, zz, ax, fig)

print(np.linalg.norm(zz))
print(np.linalg.norm(zz0))

In [ ]:
# SLSQP
constraint_v = dict()
constraint_v["type"] = "eq"
constraint_v["fun"] = lambda p: op["IDx"]@p-(cor)*op["mass_tilde_x"]@ q_vec["v"]
constraint_v["jac"] = lambda p: op["IDx"]

constraint_u = dict()
constraint_u["type"] = "eq"
constraint_u["fun"] = lambda p: op["IDy"]@p+(cor)*op["mass_tilde_y"]@ q_vec["u"]
constraint_u["jac"] = lambda p: op["IDy"]


res = minimize(target_function, p_ic, method='SLSQP', \
               jac=target_function_dir, hess=target_function_hess,
            constraints=[constraint_u, constraint_v])

print(np.linalg.norm(linear_constraint_v.residual(res.x)[0],np.inf))
print(np.linalg.norm(linear_constraint_u.residual(res.x)[0],np.inf))

q_vec["p"] = res.x


In [ ]:

res = minimize(target_function, p_ic, method='trust-constr', \
               jac=target_function_dir, hess=target_function_hess,
            constraints=[all_constraints],
            options={'verbose': 1})

print(np.linalg.norm(linear_constraint_v.residual(p_ic)[0],np.inf))
print(np.linalg.norm(linear_constraint_u.residual(p_ic)[0],np.inf))
print(np.linalg.norm(linear_constraint_v.residual(res.x)[0],np.inf))
print(np.linalg.norm(linear_constraint_u.residual(res.x)[0],np.inf))

q_vec["p"] = res.x

In [ ]:
print(op["IDx"].shape)
print(np.linalg.matrix_rank(op["IDx"].toarray()))

In [ ]:
plot_one_sol(problem,FEM2D, q_vec)

In [ ]:
np.max(np.abs(q_vec["p"]-solver.ic_vect["p"]))

## Mario version

In [ ]:
# q_vec has already div-free velocity

p0 = q_vec["p"][0]
a = 0.5

p_mat = FEM2D.vect_to_mat(q_vec["p"])
u_mat = FEM2D.vect_to_mat(q_vec["u"])
v_mat = FEM2D.vect_to_mat(q_vec["v"])

p_new = np.zeros(p_mat.shape)
Ixv   = np.zeros(p_mat.shape)
Iyu   = np.zeros(p_mat.shape)

# Computing I^x v 
for l, y in enumerate(FEM2D.xx_dofs[1]):
   # Compute I^x v(y_l) m = int x_0^x_m v(x,y_l) dx
   for j_cell in range(geom.N_elem_dir[0]):
      j_cell_global = j_cell*FEM1Dx.degree      
      j1_cell_global = (j_cell+1)*FEM1Dx.degree

      dx = geom.xx[0][j_cell+1]-geom.xx[0][j_cell]
      Ixv[ j_cell_global+1:j1_cell_global+1, l ] =Ixv[ j_cell_global,l]+\
        dx*FEM1Dx.matrix["int_mat"][1:,:]@ v_mat[ j_cell_global:j1_cell_global+1, l]
      

# Computing I^y u 
for l, x in enumerate(FEM2D.xx_dofs[0]):
   # Compute I^y u(x_l) m = int y_0^y_m u(x_l,y) dy
   for j_cell in range(geom.N_elem_dir[1]):
      j_cell_global = j_cell*FEM1Dy.degree      
      j1_cell_global = (j_cell+1)*FEM1Dy.degree

      dy = geom.xx[1][j_cell+1]-geom.xx[1][j_cell]
      Iyu[ l, j_cell_global+1:j1_cell_global+1 ] =Iyu[ l,j_cell_global]+\
        dy*FEM1Dy.matrix["int_mat"][1:,:]@ u_mat[ l, j_cell_global:j1_cell_global+1]
      
p_new = p0 + problem.coriolis*a*Ixv - problem.coriolis*(1-a)*Iyu

p_vec = FEM2D.mat_to_vect(p_new)

q_new = dict()
q_new["u"] = q_vec["u"]
q_new["v"] = q_vec["v"]
q_new["p"] = p_vec


In [ ]:

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], p_mat)
plt.colorbar()
plt.show()
plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], p_new)
plt.colorbar()
plt.show()
plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], p_new-p_mat)
plt.colorbar()
plt.show()


In [ ]:
zzz = FEM2D.operator["IDx"]@q_new["p"] - problem.coriolis*FEM2D.operator["mass_tilde_x"]@q_new["v"]

print("Max Residual of D_x M_y (p-cor*I^x v) ", np.max(np.abs(zzz)))

zzz = FEM2D.operator["IDy"]@q_new["p"] + problem.coriolis*FEM2D.operator["mass_tilde_y"]@q_new["u"]

print("Max Residual of M_x D_y (p+cor*I^y u) ", np.max(np.abs(zzz)))

zzz = q_new["p"] -q_vec["p"]
print("Max p error wrt initial condition ", np.max(np.abs(zzz)))
